In [1]:
from db_connector import *


========| ('irfan_admin', 'trade_app') |========
*** ✅ SUCCESSFUL CLOUD CONNECTION ⛓️ ***


In [2]:
df = fn_read_data_cloud("gold","bist_master_combined_indicators")


In [4]:
def calc_poc_cluster_bonus(df):
    def cluster_bonus(g):
        g_unique = g.drop_duplicates(subset=["FRVP_HIGHEST_DATE"]).copy()
        count = len(g_unique)

        if count < 2:
            return 0

        pocs = g_unique["FRVP_POC"].dropna().astype(float).values
        if len(pocs) < 2:
            return 0

        spread = (pocs.max() - pocs.min()) / pocs.min()

        if spread <= 0.02:
            if count >= 4:
                return 15
            elif count == 3:
                return 10
            elif count == 2:
                return 5

        return 0

    bonus_df = (
        df.groupby(["EXCHANGE", "SYMBOL"], group_keys=False)
        .apply(lambda g: pd.Series({"score7": cluster_bonus(g)}))
        .reset_index()
    )

    out = df.merge(bonus_df, on=["EXCHANGE", "SYMBOL"], how="left")
    return out["score7"].fillna(0).to_numpy()

In [17]:
import numpy as np
import pandas as pd

# =========================
# HELPERS
# =========================
def to_float(col):
    return (
        col.astype(str)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .astype(float)
    )

# =========================
# MFI
# =========================
def calc_mfi_new(df):
    base = (
        np.where(df["MFI"] > df["MFI_YESTERDAY"], 25, 0) +
        np.where(df["MFI"] > df["MFI_12DAY_AVG"], 50, 0) +
        np.where(df["MFI_DIRECTION"] == "Upward", 25, 0)
    )

    scaled = (base / 100) * 70

    # MFI > 80 ise +30 ekle (toplam max 100)
    final = np.where(df["MFI"] > 80, scaled + 30, scaled)

    return final

# =========================
# RSI
# =========================
def calc_rsi_new(df):
    rsi = to_float(df["RSI"])
    rsi_ma = pd.to_numeric(df["RSI_MA"], errors="coerce")
    status = pd.to_numeric(df["RSI_STATUS"], errors="coerce").fillna(0)
    days = pd.to_numeric(df["RSI_CROSS_DAYS_AGO"], errors="coerce").fillna(0)

    score1 = np.where(rsi > rsi_ma, 70, 0)

    days = days
    score2 = np.where(
        (status == 1) & (days <= 14),
        np.maximum(0, 15 - days),
        0
    )

    score3 = np.where((rsi >= 30) & (rsi <= 70), 15, 0)

    return np.clip(score1 + score2 + score3, 0, 100)

# =========================
# EMA
# =========================
def calc_ema_new(df):
    def ema_score(status, cross, days):
        status = pd.to_numeric(status, errors="coerce").fillna(0)
        cross  = pd.to_numeric(cross, errors="coerce").fillna(0)
        days   = pd.to_numeric(days, errors="coerce").fillna(0)

        return np.where(
            (status == 1) & (cross == 1),
            np.maximum(0, 3 - (days / 10)),
            0
        )

    total = (
        ema_score(df["EMA_STATUS_5_20"], df["EMA_CROSS_5_20"], df["EMA_DAYS_SINCE_CROSS_5_20"]) +
        ema_score(df["EMA_STATUS_3_20"], df["EMA_CROSS_3_20"], df["EMA_DAYS_SINCE_CROSS_3_20"]) +
        ema_score(df["EMA_STATUS_3_14"], df["EMA_CROSS_3_14"], df["EMA_DAYS_SINCE_CROSS_3_14"])
    )

    return np.clip((total / 9) * 100, 0, 100)

# =========================
# VOLUME
# =========================
def calc_volume_new(df):
    vol_last = to_float(df["VOL_LASTDAY"])
    vol_yest = to_float(df["VOL_YESTERDAY"])
    vol_5 = to_float(df["VOL_AVG_5DAY"])
    vol_10 = to_float(df["VOL_AVG_10DAY"])
    vol_20 = to_float(df["VOL_AVG_20DAY"])

    return (
        np.where(vol_last > vol_yest, 25, 0) +
        np.where(vol_last > vol_5, 25, 0) +
        np.where(vol_5 > vol_10, 25, 0) +
        np.where(vol_5 > vol_20, 25, 0)
    )

# =========================
# VWAP
# =========================
def calc_vwap_new(df):
    close = df["FRVP_LATEST_CLOSE_VALUE"]
    vwap = df["VWAP"]

    dist = np.abs(close - vwap) / vwap

    return np.clip(
        np.where(dist <= 0.05, 0,
                 np.where(dist >= 0.10, 100,
                          ((dist - 0.05) / 0.05) * 100)),
        0, 100
    )

# =========================
# POC FRVP
# =========================
import numpy as np
import pandas as pd

def calc_poc_frvp_new(df, log_file="poc_log.txt"):

    close = df["FRVP_LATEST_CLOSE_VALUE"].astype(float)
    poc = df["FRVP_POC"].astype(float)
    val = df["FRVP_VAL"].astype(float)
    vah = df["FRVP_VAH"].astype(float)

    # =========================
    # VALUE AREA WIDTH
    # =========================
    value_range = vah - val

    # =========================
    # 1 → CLOSE vs POC (10 puan)
    # =========================
    dist_close_poc = np.abs(close - poc)

    score1 = np.where(
        dist_close_poc <= (value_range * 0.05),
        10,
        np.where(
            dist_close_poc <= (value_range * 0.80),
            (1 - (dist_close_poc - value_range * 0.05) / (value_range * 0.75)) * 10,
            0
        )
    )

    score1 = np.clip(score1, 0, 10)

    # =========================
    # 2 → CLUSTER BONUS (15 puan)
    # =========================
    score2 = calc_poc_cluster_bonus(df)

    # =========================
    # 3 → GREEN (5 puan)
    # =========================
    score3 = np.where(df["BS_BAR_STATUS"] == "GREEN", 5, 0)

    # =========================
    # 4 → CONTACTLESS (5 puan)
    # =========================
    score4 = np.where(
        (df["BS_OPEN_PRICE"].astype(float) > poc) &
        (df["BS_BAR_STATUS"] == "GREEN"),
        5,
        0
    )

    # =========================
    # 5 → POC vs VAL (5 puan)
    # =========================
    dist_poc_val = np.abs(poc - val)

    score5 = np.where(
        dist_poc_val <= (value_range * 0.10),
        5,
        np.where(
            dist_poc_val <= (value_range * 0.40),
            (1 - (dist_poc_val - value_range * 0.10) / (value_range * 0.30)) * 5,
            0
        )
    )

    score5 = np.clip(score5, 0, 5)

    # =========================
    # TOTAL (max = 40)
    # =========================
    total = score1 + score2 + score3 + score4 + score5

    final_score = np.clip((total / 40) * 100, 0, 100)

    # =========================
    # LOG
    # =========================
    with open(log_file, "w", encoding="utf-8") as f:

        for i in range(len(df)):

            f.write(f"""
SYMBOL: {df.iloc[i]["SYMBOL"]}
PERIOD: {df.iloc[i]["FRVP_PERIOD_TYPE"]}
HIGHEST_DATE: {df.iloc[i]["FRVP_HIGHEST_DATE"]}
GUN: {df.iloc[i]["CREATED_DAY"]}

VAL: {val.iloc[i]:.2f} | POC: {poc.iloc[i]:.2f} | VAH: {vah.iloc[i]:.2f}
CLOSE: {close.iloc[i]:.2f}

score1 (close→poc): {score1[i]:.2f}
score2 (cluster): {score2[i]}
score3 (green): {score3[i]}
score4 (contactless): {score4[i]}
score5 (poc→val): {score5[i]:.2f}

TOTAL: {total[i]:.2f} → FINAL: {final_score[i]:.2f}
-------------------------------------------
""")

    return final_score

# =========================
# TRADE LEVELS
# =========================

def calculate_trade_levels(df):

    close = df["FRVP_LATEST_CLOSE_VALUE"]
    df["entry_price"] = close * 1.005

    pivot = df["PIVOT"] if "PIVOT" in df.columns else pd.Series(np.nan, index=df.index)
    r2 = df["R2"] if "R2" in df.columns else pd.Series(np.nan, index=df.index)

    def choose_target(row):

        c = row["FRVP_LATEST_CLOSE_VALUE"]

        if c < row["FRVP_POC"]:
            return row["VWAP"]

        elif c < row["VWAP"]:
            return row["VWAP"]

        elif c < row["FRVP_VAH"]:
            return row["FRVP_VAH"]

        elif pd.notna(pivot.loc[row.name]) and c < pivot.loc[row.name]:
            return pivot.loc[row.name]

        elif pd.notna(r2.loc[row.name]):
            return r2.loc[row.name]

        else:
            return row["FRVP_VAH"]

    df["target_price"] = df.apply(choose_target, axis=1)

    df["target_pct"] = ((df["target_price"] - df["entry_price"]) / df["entry_price"]) * 100
    df["risk_pct"] = ((df["entry_price"] - df["stop_loss"]) / df["entry_price"]) * 100
    df["rr_ratio"] = df["target_pct"] / df["risk_pct"]

    df["pivot_display"] = np.where(
        pivot.isna(),
        "N/A",
        pivot
    )

    return df

# =========================
# MASTER FUNCTION
# =========================

def calculate_scores(df):

    df["poc_frvp_status"] = calc_poc_frvp_new(df)
    df["vwap_status"] = calc_vwap_new(df)
    df["ema_status"] = calc_ema_new(df)
    df["rsi_status"] = calc_rsi_new(df)
    df["mfi_status"] = calc_mfi_new(df)
    df["vol_status"] = calc_volume_new(df)

    # =========================
    # MASTER SCORE
    # =========================
    df["master_score"] = (
        df["poc_frvp_status"] * 0.62 +
        df["vwap_status"] * 0.02 +
        df["ema_status"] * 0.09 +
        df["rsi_status"] * 0.07 +
        df["mfi_status"] * 0.10 +
        df["vol_status"] * 0.10
    )

    df["master_score"] = np.clip(df["master_score"], 0, 100)

    # =========================
    # WATCHLIST
    # =========================
    df["watchlist"] = np.where(
        df["master_score"] >= 70, "BUY",
        np.where(df["master_score"] >= 50, "WATCH", "AVOID")
    )
    df["entry_price"] = df["FRVP_LATEST_CLOSE_VALUE"] * 1.005
    

    # =========================
    # STOP LOSS (ENTRY’den sonra!)
    # =========================
    df["stop_loss"] = df["entry_price"] * 0.97

    # =========================
    # TRADE LEVELS (ENTRY burada oluşuyor)
    # =========================
    df = calculate_trade_levels(df)

    return df

In [18]:
scores = calculate_scores(df)


# --- TRIAGE KISMI----

In [19]:
def calculate_triage_selection(df):

    # df_filtered = df[df["BS_CLOSE_PRICE"] > df["FRVP_POC"]].copy()
    df_filtered = df[
    (df["BS_CLOSE_PRICE"] > df["FRVP_POC"])].copy()

    if df_filtered.empty:
        return pd.DataFrame()

    grouped = df_filtered.groupby(["EXCHANGE", "SYMBOL"])

    results = []

    for (exchange, symbol), g in grouped:

        # =========================
        # UNIQUE CLUSTER COUNT
        # =========================
        g_unique = g.drop_duplicates(subset=["FRVP_HIGHEST_DATE"])
        count = len(g_unique)

        # =========================
        # SCORE
        # =========================
        avg_score = g_unique["master_score"].mean()

        # =========================
        # POC CLUSTER ANALYSIS
        # =========================
        pocs = g_unique["FRVP_POC"].values

        poc_counts = g["FRVP_POC"].value_counts()

        max_freq = poc_counts.max()

        dominant_pocs = poc_counts[poc_counts == max_freq].index

        avg_poc = np.mean(dominant_pocs)

        # =========================
        # TRADE LEVELS (UPDATED)
        # =========================

        last_close = g["BS_CLOSE_PRICE"].iloc[-1]

        # STOP LOSS → %3 aşağı
        stop_loss_price = last_close * (1 - 0.03)

        # TARGET → mevcut avg_target yerine relative hesap
        avg_target = g["target_price"].mean()

        target_pct = ((avg_target - last_close) / last_close) * 100

        triage_day = pd.to_datetime(g["CREATED_DAY"].iloc[0], dayfirst=True).strftime("%Y-%m-%d")

        results.append({
            "EXCHANGE": exchange,
            "SYMBOL": symbol,
            "triage_entry_day": triage_day,

            "triage_score": round(min(100, avg_score), 2),
            "valid_cluster_count": count,

            "avg_poc": f"{avg_poc:.2f}".replace(".", ","),

            # ✅ yeni yapı
            "stop_loss": f"{stop_loss_price:.2f}".replace(".", ","),
            "target_price": f"{avg_target:.2f}".replace(".", ","),

            # ✅ yüzde artık dinamik
            "target_%": f"{target_pct:.2f}%".replace(".", ",")
        })

    result_df = pd.DataFrame(results)

    result_df = result_df.sort_values("triage_score", ascending=False)

    return result_df


In [20]:
def build_final_triage_df(df):

    triage_df = calculate_triage_selection(df)

    if triage_df.empty:
        return pd.DataFrame()

    # rank
    triage_df["rank"] = (
        triage_df
        .groupby("triage_entry_day")["triage_score"]
        .rank(method="first", ascending=False)
        .astype(int)
    )

    # merge key
    df["triage_entry_day"] = pd.to_datetime(df["CREATED_DAY"], dayfirst=True).dt.strftime("%Y-%m-%d")

    merged = df.merge(
        triage_df,
        on=["EXCHANGE", "SYMBOL", "triage_entry_day"],
        how="left"
    )

    # =========================
    # YENİ KOLONLAR
    # =========================
    merged["MASTER_SCORE"] = merged["master_score"].round(2)

    merged["RSI_STATUS_TXT"] = merged["rsi_status"]
    merged["MFI_STATUS_TXT"] = merged["mfi_status"]
    merged["VOL_STATUS_TXT"] = merged["vol_status"]

    merged["STOP_LOSS_PERC"] = "-3,00%"
    merged["TARGET_PERC"] = merged.get("target_%", None)

    merged["RUNTIME"] = np.nan

    # rename triage
    merged.rename(columns={
        "triage_score": "TRIAGE_SCORE",
        "valid_cluster_count": "VALID_CLUSTER_COUNT",
        "avg_poc": "AVG_POC",
        "stop_loss_y": "STOP_LOSS",
        "target_price_y": "TARGET_PRICE",
        "rank": "RANK",
        "triage_entry_day": "TRIAGE_ENTRY_DAY"
    }, inplace=True)

    # =========================
    # FINAL KOLONLAR (senin verdiğin sıraya göre)
    # =========================
    final_cols = [
        "EXCHANGE","SYMBOL","FRVP_INTERVAL","FRVP_PERIOD_TYPE",
        "FRVP_HIGHEST_DATE","FRVP_HIGHEST_VALUE",
        "FRVP_ROW_COUNT_AFTER_HIGHEST","FRVP_DAY_COUNT_AFTER_HIGHEST",
        "FRVP_LATEST_CLOSE_VALUE","FRVP_POC","FRVP_VAL","FRVP_VAH",
        "BS_OPEN_PRICE","BS_CLOSE_PRICE","BS_DIFFER","BS_PERC","BS_BAR_STATUS",
        "RAW_END_DATE","BRONZE_END_DATE","SILVER_END_DATE","SILVER_CONVERTED_END_DATE","EXPECTED_END_DATE",
        "EMA_TIMESTAMP","EMA_END_DATE","EMA3","EMA5","EMA14","EMA20",
        "EMA_STATUS_5_20","EMA_CROSS_5_20",
        "EMA_STATUS_3_20","EMA_CROSS_3_20",
        "EMA_STATUS_3_14","EMA_CROSS_3_14",
        "EMA_DAYS_SINCE_CROSS_5_20","EMA_DAYS_SINCE_CROSS_3_20","EMA_DAYS_SINCE_CROSS_3_14",
        "RSI","RSI_MA","RSI_STATUS","RSI_CROSS","RSI_CROSS_DAYS_AGO",
        "MFI","MFI_YESTERDAY","MFI_12DAY_AVG","MFI_DIRECTION",
        "VWAP_HIGHEST_VALUE","VWAP_HIGHEST_TIMESTAMP","VWAP",
        "VOL_AVG_5DAY","VOL_AVG_10DAY","VOL_AVG_20DAY",
        "VOL_YESTERDAY","VOL_LASTDAY","VOL_STATUS",
        "PVT_START_DATE","PVT_END_DATE","PVT_YEAR",
        "PIVOT","PVT_R1","PVT_R2","PVT_R3","PVT_R4","PVT_R5",
        "PVT_S1","PVT_S2","PVT_S3","PVT_S4","PVT_S5","PVT_STATUS",

        "poc_frvp_status","vwap_status","ema_status",
        "RSI_STATUS_TXT","MFI_STATUS_TXT","VOL_STATUS_TXT",

        "MASTER_SCORE",

        "TRIAGE_ENTRY_DAY","TRIAGE_SCORE","RANK",
        "VALID_CLUSTER_COUNT","AVG_POC",
        "STOP_LOSS","TARGET_PRICE","STOP_LOSS_PERC","TARGET_PERC",

        "CREATED_AT","CREATED_DAY","RUNTIME"
    ]
    final_cols = [c for c in final_cols if c in merged.columns]

    return merged[final_cols]

In [21]:
son_hali = build_final_triage_df(scores)


In [22]:
pd.set_option('display.max_columns', None)

In [23]:
son_hali

,EXCHANGE,SYMBOL,FRVP_INTERVAL,FRVP_PERIOD_TYPE,FRVP_HIGHEST_DATE,FRVP_HIGHEST_VALUE,FRVP_ROW_COUNT_AFTER_HIGHEST,FRVP_DAY_COUNT_AFTER_HIGHEST,FRVP_LATEST_CLOSE_VALUE,FRVP_POC,FRVP_VAL,FRVP_VAH,BS_OPEN_PRICE,BS_CLOSE_PRICE,BS_DIFFER,BS_PERC,BS_BAR_STATUS,RAW_END_DATE,BRONZE_END_DATE,SILVER_END_DATE,SILVER_CONVERTED_END_DATE,EXPECTED_END_DATE,EMA_TIMESTAMP,EMA_END_DATE,EMA3,EMA5,EMA14,EMA20,EMA_STATUS_5_20,EMA_CROSS_5_20,EMA_STATUS_3_20,EMA_CROSS_3_20,EMA_STATUS_3_14,EMA_CROSS_3_14,EMA_DAYS_SINCE_CROSS_5_20,EMA_DAYS_SINCE_CROSS_3_20,EMA_DAYS_SINCE_CROSS_3_14,RSI,RSI_MA,RSI_STATUS,RSI_CROSS,RSI_CROSS_DAYS_AGO,MFI,MFI_YESTERDAY,MFI_12DAY_AVG,MFI_DIRECTION,VWAP_HIGHEST_VALUE,VWAP_HIGHEST_TIMESTAMP,VWAP,VOL_AVG_5DAY,VOL_AVG_10DAY,VOL_AVG_20DAY,VOL_YESTERDAY,VOL_LASTDAY,VOL_STATUS,PVT_START_DATE,PVT_END_DATE,PVT_YEAR,PIVOT,PVT_R1,PVT_R2,PVT_R3,PVT_R4,PVT_R5,PVT_S1,PVT_S2,PVT_S3,PVT_S4,PVT_S5,PVT_STATUS,poc_frvp_status,vwap_status,ema_status,RSI_STATUS_TXT,MFI_STATUS_TXT,VOL_STATUS_TXT,MASTER_SCORE,TRIAGE_ENTRY_DAY,TRIAGE_SCORE,RANK,VALID_CLUSTER_COUNT,AVG_POC,STOP_LOSS,TARGET_PRICE,STOP_LOSS_PERC,TARGET_PERC,CREATED_AT,CREATED_DAY,RUNTIME
0,BIST,USAK,hourly,3months,2026-01-26 07:00:00,2.252584,584,59,1.83,2.252583,1.692657,2.252583,1.76,1.83,0.07,3.98,GREEN,2026-04-17 17:00:00,2026-04-17 17:00:00,2026-04-17 17:00:00,2026-04-17,2026-04-17,2026-04-17,2026-04-17,1.784935,1.759630,1.703146,1.689666,1,0,1,0,1,0,7,9,9,65.262642,51.979839,1,0,12.0,88.048345,84.734964,73.583750,Upward,2.252584,2026-01-26 07:00:00,1.81,133658999.2,169343155.4,1.209723e+08,158005060.0,145750545.0,negative,2025-04-17,2026-04-17,2026.0,2.913790,2.328847,2.606834,2.88482,3.718781,5.451367,1.772873,1.494886,1.21690,0.382939,-1.349647,PASSED,14.009592,0.000000,0.000000,73.0,100.0,50,28.80,2026-04-20,NaN,NaN,NaN,NaN,NaN,NaN,"-3,00%",NaN,2026-04-20 13:26:24.732229,20-04-2026,NaN
1,BIST,USAK,hourly,6months,2025-11-04 15:00:00,2.608963,1146,116,1.83,2.440860,1.824986,2.481206,1.76,1.83,0.07,3.98,GREEN,2026-04-17 17:00:00,2026-04-17 17:00:00,2026-04-17 17:00:00,2026-04-17,2026-04-17,2026-04-17,2026-04-17,1.784935,1.759630,1.703146,1.689666,1,0,1,0,1,0,7,9,9,65.262642,51.979839,1,0,12.0,88.048345,84.734964,73.583750,Upward,2.608963,2025-11-04 15:00:00,2.04,133658999.2,169343155.4,1.209723e+08,158005060.0,145750545.0,negative,2025-04-17,2026-04-17,2026.0,2.913790,2.328847,2.606834,2.88482,3.718781,5.451367,1.772873,1.494886,1.21690,0.382939,-1.349647,PASSED,12.500000,100.000000,0.000000,73.0,100.0,50,29.86,2026-04-20,NaN,NaN,NaN,NaN,NaN,NaN,"-3,00%",NaN,2026-04-20 13:26:24.732229,20-04-2026,NaN
2,BIST,USAK,hourly,1year,2025-05-09 10:00:00,4.861547,2360,238,1.83,2.440860,1.552643,3.120230,1.76,1.83,0.07,3.98,GREEN,2026-04-17 17:00:00,2026-04-17 17:00:00,2026-04-17 17:00:00,2026-04-17,2026-04-17,2026-04-17,2026-04-17,1.784935,1.759630,1.703146,1.689666,1,0,1,0,1,0,7,9,9,65.262642,51.979839,1,0,12.0,88.048345,84.734964,73.583750,Upward,4.861547,2025-05-09 10:00:00,2.75,133658999.2,169343155.4,1.209723e+08,158005060.0,145750545.0,negative,2025-04-17,2026-04-17,2026.0,2.913790,2.328847,2.606834,2.88482,3.718781,5.451367,1.772873,1.494886,1.21690,0.382939,-1.349647,PASSED,26.177276,100.000000,0.000000,73.0,100.0,50,38.34,2026-04-20,NaN,NaN,NaN,NaN,NaN,NaN,"-3,00%",NaN,2026-04-20 13:26:24.732229,20-04-2026,NaN
3,BIST,USAK,hourly,2year,2024-07-25 08:00:00,5.581029,4346,437,1.83,3.001652,1.745214,3.483100,1.76,1.83,0.07,3.98,GREEN,2026-04-17 17:00:00,2026-04-17 17:00:00,2026-04-17 17:00:00,2026-04-17,2026-04-17,2026-04-17,2026-04-17,1.784935,1.759630,1.703146,1.689666,1,0,1,0,1,0,7,9,9,65.262642,51.979839,1,0,12.0,88.048345,84.734964,73.583750,Upward,5.581029,2024-07-25 08:00:00,2.56,133658999.2,169343155.4,1.209723e+08,158005060.0,145750545.0,negative,2025-04-17,2026-04-17,2026.0,2.913790,2.328847,2.606834,2.88482,3.718781,5.451367,1.772873,1.494886,1.21690,0.382939,-1.349647,PASSED,16.693923,100.000000,0.000000,73.0,100.0,50,32.46,2026-04-20,NaN,NaN,NaN,NaN,NaN,NaN,"-3,00%",NaN